# Transfer Learning: Sentiment Analysis with DistilBERT

Fine-tuning a pretrained **DistilBERT** transformer for 3-class sentiment classification (negative / neutral / positive).

**Dataset:** The [Sentiment Analysis Dataset](https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset) from Kaggle. Of the four CSV files in the archive, we use `train.csv` and `test.csv`.

**Pipeline:** upload CSVs → clean → encode labels → tokenize with DistilBERT tokenizer → fine-tune `DistilBertForSequenceClassification` (3 output classes) → evaluate.

## Setup

In [1]:
!pip install transformers -q

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import classification_report

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Step 1 — Upload the CSV files

Running this cell opens a file picker. I select **both** `train.csv` and `test.csv` from my computer.

In [3]:
from google.colab import files
uploaded = files.upload()
print("\nUploaded:", list(uploaded.keys()))

Saving test.csv to test.csv
Saving train.csv to train.csv

Uploaded: ['test.csv', 'train.csv']


## Step 2 — Load and clean the data

Load both CSVs (the files are latin-1 encoded), keep only the `text` and `sentiment` columns, drop rows with missing text, and encode labels as integers: `negative=0`, `neutral=1`, `positive=2`.

In [4]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}
label_names = ["negative", "neutral", "positive"]

def load_clean(path):
    df = pd.read_csv(path, encoding="latin-1")[["text", "sentiment"]]
    df = df.dropna(subset=["text", "sentiment"])
    df = df[df["text"].str.strip().astype(bool)]
    df = df[df["sentiment"].isin(label_map.keys())]
    df["label"] = df["sentiment"].map(label_map)
    return df.reset_index(drop=True)

train_df = load_clean("train.csv")
test_df = load_clean("test.csv")

print(f"Train: {len(train_df)} rows")
print(f"Test:  {len(test_df)} rows")
print(f"\nTrain class distribution:\n{train_df['sentiment'].value_counts()}")
print(f"\nTest class distribution:\n{test_df['sentiment'].value_counts()}")

Train: 27480 rows
Test:  3534 rows

Train class distribution:
sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64

Test class distribution:
sentiment
neutral     1430
positive    1103
negative    1001
Name: count, dtype: int64


## Step 3 — Tokenize and build PyTorch Datasets

DistilBERT requires text to be converted to token IDs. We use its fast tokenizer with `max_length=128`. The custom `Dataset` class returns `input_ids`, `attention_mask`, and `labels` for each example.

In [5]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }

train_dataset = SentimentDataset(train_df["text"], train_df["label"], tokenizer, MAX_LEN)
test_dataset = SentimentDataset(test_df["text"], test_df["label"], tokenizer, MAX_LEN)
print(f"Tokenized — train: {len(train_dataset)}, test: {len(test_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenized — train: 27480, test: 3534


## Step 4 — DataLoaders

In [6]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 859 | Test batches: 111


## Step 5 — Load pretrained DistilBERT

Load `DistilBertForSequenceClassification` with `num_labels=3`. The pretrained transformer body provides general language representations; a fresh classification head is attached on top and will be trained alongside (full fine-tuning of all parameters).

In [7]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params: 66,955,779


## Step 6 — Train

Fine-tune for 3 epochs with learning rate 2e-5.

In [8]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/total:.4f} - Acc: {correct/total:.4f}")

Epoch 1/3 - Loss: 0.5877 - Acc: 0.7544
Epoch 2/3 - Loss: 0.4367 - Acc: 0.8284
Epoch 3/3 - Loss: 0.3301 - Acc: 0.8755


## Step 7 — Evaluate

In [9]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=label_names))

              precision    recall  f1-score   support

    negative       0.78      0.79      0.78      1001
     neutral       0.76      0.75      0.76      1430
    positive       0.84      0.83      0.84      1103

    accuracy                           0.79      3534
   macro avg       0.79      0.79      0.79      3534
weighted avg       0.79      0.79      0.79      3534

